# 01 — Acquire Riemann zero data

Download and validate the selected Odlyzko zero dataset.

In [ ]:
from pathlib import Path
import hashlib
import urllib.request

DATA_DIR = Path("data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

ODLYZKO_BASE = "https://www-users.cse.umn.edu/~odlyzko/zeta_tables"
DATASET = "zeros1"
URL = f"{ODLYZKO_BASE}/{DATASET}"
RAW_FILE = DATA_DIR / DATASET

print("URL :", URL)
print("file:", RAW_FILE)

In [ ]:
if RAW_FILE.exists():
    print(f"Already exists: {RAW_FILE}")
    print(f"Bytes: {RAW_FILE.stat().st_size:,}")
else:
    urllib.request.urlretrieve(URL, RAW_FILE)
    print(f"Downloaded: {RAW_FILE}")
    print(f"Bytes: {RAW_FILE.stat().st_size:,}")

In [ ]:
def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

checksum = sha256(RAW_FILE)
print("SHA-256:", checksum)

In [ ]:
with RAW_FILE.open("r", encoding="ascii") as f:
    for _ in range(10):
        print(repr(f.readline()))

In [ ]:
with RAW_FILE.open("r", encoding="ascii") as f:
    lines = f.readlines()

print("lines:", len(lines))
print("first:", repr(lines[0]))
print("last :", repr(lines[-1]))

In [ ]:
import numpy as np

gamma = np.loadtxt(RAW_FILE, dtype=np.float64)

assert gamma.ndim == 1
assert np.all(np.isfinite(gamma))
assert np.all(np.diff(gamma) > 0)

print("N =", len(gamma))
print("first =", gamma[:5])
print("last  =", gamma[-5:])

In [ ]:
from nicht_riemann_data.transforms import spacings
from nicht_riemann_data.diagnostics import describe

delta = spacings(gamma)

assert len(delta) == len(gamma) - 1
assert np.all(np.isfinite(delta))
assert np.all(delta > 0)

print("zeros    :", len(gamma))
print("spacings :", len(delta))
print("first    :", delta[:10])

print("gamma :")
describe(gamma)
print("delta :")
describe(delta)

In [ ]:
raw_bytes = RAW_FILE.stat().st_size

print("raw bytes    :", raw_bytes)
print("zeros        :", len(gamma))
print("bytes / zero :", raw_bytes / len(gamma))